In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
})

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5.RDS")

In [ ]:
cds_VCS$Cluster = clusters(cds_VCS)

In [ ]:
colnames(colData(cds_VCS))

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_VCS, color_cells_by = "cluster", x = 1, y = 2, cell_size = 0.1)
plot_cells(cds_VCS, color_cells_by = "cluster", x = 1, y = 3, cell_size = 0.1)
plot_cells(cds_VCS, color_cells_by = "cluster", x = 2, y = 3, cell_size = 0.1)

In [ ]:
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_VCS, color_cells_by = "nn_majority_cell_type_pca", x = 1, y = 2, cell_size = 0.1)
plot_cells(cds_VCS, color_cells_by = "nn_majority_cell_type_pca", x = 1, y = 3, cell_size = 0.1)
plot_cells(cds_VCS, color_cells_by = "nn_majority_cell_type_pca", x = 2, y = 3, cell_size = 0.1)

In [ ]:
as.data.frame(colData(cds_VCS)) %>%
    group_by(Cluster, nn_majority_trajectory_pca) %>%
    count() %>%
    group_by(Cluster) %>%
    arrange(Cluster, -n) %>%
    distinct(Cluster, .keep_all=T) %>%
    as.data.frame() %>%
    select(Cluster, nn_majority_trajectory_pca) %>%
    count(nn_majority_trajectory_pca)

In [ ]:
trajectories = as.data.frame(colData(cds_VCS)) %>%
    select(nn_majority_trajectory_pca) %>%
    unique() %>% pull(nn_majority_trajectory_pca)

In [ ]:
as.data.frame(colData(cds_VCS)) %>%
    group_by( nn_majority_trajectory_pca) %>%
    count() %>% arrange(-n)

In [ ]:
as.data.frame(colData(cds_VCS)) %>%
    filter(nn_majority_trajectory_pca == "Adipocytes") %>%
    group_by(Cluster, nn_majority_trajectory_pca) %>%
    count() %>% arrange(-n)

In [ ]:
options(repr.plot.width=5, repr.plot.height=5, repr.plot.res=200)

for (trajectory in trajectories) {
    g = list()
    
    g[[1]] = plot_cells(cds_VCS[, cds_VCS$nn_majority_trajectory_pca == trajectory], group_label_size=3,
               color_cells_by = "cluster", x = 1, y = 2) + 
                ggtitle(trajectory)
    
    g[[2]] = plot_cells(cds_VCS[, cds_VCS$nn_majority_trajectory_pca == trajectory], group_label_size=3,
               color_cells_by = "cluster", x = 1, y = 3) + 
                ggtitle(trajectory)
    
    g[[3]] = plot_cells(cds_VCS[, cds_VCS$nn_majority_trajectory_pca == trajectory], group_label_size=3,
               color_cells_by = "cluster", x = 2, y = 3) + 
                ggtitle(trajectory)
    
    print(g[[1]])
    print(g[[2]])
    print(g[[3]])
}

In [ ]:
cluster_assignment_list = list()
cluster_assignment_list[['Mesoderm']] = c(7, 9, 11, 24, 25, 36, 38, 46, 49, 51, 55, 56, 59)
cluster_assignment_list[['Endothelium']] = c(26)
cluster_assignment_list[['Muscle_cells']] = c(14, 41, 48)
cluster_assignment_list[['Adipocytes']] = c(9, 25, 55)
cluster_assignment_list[['Hepatocytes']] = c(13, 54, 63)
cluster_assignment_list[['Definitive_erythroid']] = c(2, 4, 5, 6, 20, 30, 37)
cluster_assignment_list[['CNS_neurons']] = c(1, 3, 8, 10, 12, 15, 18, 21, 27, 32, 35, 42) #
cluster_assignment_list[['Epithelium']] = c(28, 31, 40, 45, 58)
cluster_assignment_list[['Intermediate_progenitors']] = c(17, 19, 22, 29, 34, 39)
cluster_assignment_list[['Brain_and_spinal_cord']] = c(12, 16, 34, 35, 42, 47, 52)
cluster_assignment_list[['B_cells']] = c(23)
cluster_assignment_list[['White_blood_cells']] = c(23, 33, 43)
cluster_assignment_list[['Megakaryocytes']] = c(23, 60)
cluster_assignment_list[['Olfactory_sensory_neurons']] = c(50) 
cluster_assignment_list[['Neural_crest_PNS_glia']] = c(12, 16, 31, 44, 47)  #
cluster_assignment_list[['Mast_cells']] = c(53)
cluster_assignment_list[['Eye_and_other']] = c(16, 31, 32, 35, 47, 50)
cluster_assignment_list[['Neural_crest_PNS_neurons']] = c(32, 35)
cluster_assignment_list[['Lung_and_airway']] = c(28, 31)
cluster_assignment_list[['T_cells']] = c(23)
cluster_assignment_list[['Ependymal_cells']] = c(16, 61)
cluster_assignment_list[['Oligodendrocytes']] = c(16, 57)
cluster_assignment_list[['Intestine']] = c(45, 58)
cluster_assignment_list[['Cardiomyocytes']] = c(41)
cluster_assignment_list[['Testis_and_adrenal']] = c(40)

In [ ]:
as.data.frame(colData(cds_VCS)) %>%
    filter(nn_majority_trajectory_pca == "Neural_crest_PNS_glia") %>%
    group_by(Cluster, nn_majority_trajectory_pca) %>%
    count() %>% arrange(-n)

In [ ]:
length(cluster_assignment_list)
metadata = as.data.frame(colData(cds_VCS))

In [ ]:
cell_df = data.frame(Cell = cds_VCS$Cell, Trajectory_cluster_refined = "Mislabeled")
for (trajectory in names(cluster_assignment_list)) {
    metadata_sub = metadata[metadata$nn_majority_trajectory_pca == trajectory & 
          metadata$Cluster %in% cluster_assignment_list[[trajectory]], ]
    cell_df[cell_df$Cell %in% metadata_sub$Cell, ]$Trajectory_cluster_refined = trajectory
}

print(dim(cell_df))
print(dim(metadata))

In [ ]:
metadata_sub = metadata[metadata$nn_majority_trajectory_pca == "Primitive_erythroid", ]
cell_df[cell_df$Cell %in% metadata_sub$Cell, ]$Trajectory_cluster_refined = "Definitive_erythroid"

In [ ]:
head(as.data.frame(colData(cds_VCS)))
head(cell_df[match(cds_VCS$Cell, cell_df$Cell),])

In [ ]:
cds_VCS$Trajectory_cluster_refined = cell_df$Trajectory_cluster_refined
head(colData(cds_VCS))

In [ ]:
saveRDS(cds_VCS, "cds_VCS_with_cell_type_labels_cluster_corrected_E15-15.5.RDS")